In [0]:
from pyspark.sql import functions as F
schema_silver = "workspace.silver"
schema_gold = "workspace.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_gold}")

DataFrame[]

In [0]:
from pyspark.sql.window import Window
#le a tabela da silver
df_info = spark.table(f"{schema_silver}.tb_info_filmes")
#ordenando por id_filme
janela = Window.orderBy("id_filme")
# Numera as linhas 1, 2, 3... e converte pra BIGINT
df_info = df_info.withColumn(
    "sk_movie_id",
    F.row_number().over(janela).cast("bigint")
)
display(df_info)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id_filme,titulo,titulo_original,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,sk_movie_id
1000004,Purple Beatz,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.",A drum n bass romance.,1
1000005,Aisha Brown: The First Black Woman Ever,Aisha Brown: The First Black Woman Ever,2020-02-14,2020,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs.",null,2
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,Kyle Brownrigg: Introducing Lyle,2022-05-27,2022,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle.",null,3
1000011,Worth Your Weight in Gold,O Teu Peso Em Ouro,2022-07-14,2022,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness.",null,4
1000014,On va manquer !,On va manquer !,2018-05-15,2018,0,fr,Lançado,null,null,5
1000030,58 Hours: The Baby Jessica Story,58 Hours: The Baby Jessica Story,2021-07-31,2021,0,es,Lançado,null,null,6
1000054,One Hundred Years and Hope,百年と希望,2022-06-18,2022,107,ja,Lançado,"In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope.",null,7
1000058,Homecoming,Le retour,2023-07-12,2023,110,fr,Lançado,"Kheìdidja, in her forties, works for a wealthy Parisian family who offers her the opportunity to take care of their children for a summer in Corsica. It's an opportunity for her to return with her daughters, Jessica and Farah, to the island they left fifteen years earlier in tragic circumstances.",null,8
1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,2016-04-05,2016,116,ja,Lançado,"Edawakare, the driver of the Time Taxi that allows passengers to return to their life's turning points, visits a hot spring this time around. An array of guests at the inn are fraught with troubles in their life and become passengers of the Time Taxi. And somehow all the clients are actually part of a bigger picture?!",null,9
1000073,A Chance To Win,Pour l'honneur,2023-05-03,2023,97,fr,Lançado,"Two villages in the south of France have always been bitter rivals, but when a group of asylum seekers arrive in the community, the life of both villages is shaken up and age-old disagreements escalate. Their antagonism reaches its peak with the annual rugby derby played between the two village teams, but this time, with the new outsiders joining as unexpected recruits, the result of the 100th match will be more unpredictable than ever.",null,10


In [0]:
# dim_movies: colunas e tipos exatamente como o documento pede
dim_movies = df_info.select(
    F.col("sk_movie_id").cast("bigint"),
    F.col("id_filme").cast("string"),
    F.col("titulo").cast("string"),
    F.col("data_lancamento").cast("date"),
    F.col("ano_lancamento").cast("int"),
    F.col("duracao_minutos").cast("int"),
    F.col("idioma_original").cast("string"),
    F.col("status_filme").cast("string"),
    F.col("sinopse").cast("string"),
)

dim_movies.write.mode("overwrite").saveAsTable(f"{schema_gold}.dim_movies")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
#fact_movies_performance

base = (spark.table(f"{schema_gold}.dim_movies").filter(F.col("status_filme")=="Lançado").select("sk_movie_id", "id_filme"))

financeiro = spark.table(f"{schema_silver}.tb_financeiro_filmes")
metricas = spark.table(f"{schema_silver}.tb_metricas_engajamento")

fato =(
    base.join(financeiro, on="id_filme", how="left")
    .join(metricas, on="id_filme", how = "left")
)

print("linhas:", fato.count())
print("filmes distintos:", fato.select("sk_movie_id").distinct().count())

linhas: 96261
filmes distintos: 96261


In [0]:
fact_movies_performance = fato.select(
    F.col("sk_movie_id").cast("bigint"),

    # financeiro, uso de decimal pra valor exato
    F.col("orcamento_usd").cast("decimal(18,2)"),
    F.col("receita_usd").cast("decimal(18,2)"),
    F.col("lucro_usd").cast("decimal(18,2)"),
    F.col("orcamento_brl").cast("decimal(18,2)"),
    F.col("receita_brl").cast("decimal(18,2)"),
    F.col("lucro_brl").cast("decimal(18,2)"),

    # engajamento
    F.col("popularidade").cast("double"),
    F.col("nota_media_tmdb").cast("double"),
    F.col("qtd_votos_tmdb").cast("int"),
    F.col("nota_media_imdb").cast("double"),
    F.col("qtd_votos_imdb").cast("int"),
)

fact_movies_performance.write.mode("overwrite").saveAsTable(
    f"{schema_gold}.fact_movies_performance"
)

In [0]:
generos = spark.table(f"{schema_silver}.tb_generos")

# catalogo ,um registro por gênero
catalogo_generos = (
    generos
        .select(F.col("nome_genero"))
        .filter(F.col("nome_genero").isNotNull())
        .distinct()
)

janela_genero = Window.orderBy("nome_genero")

dim_genres = catalogo_generos.select(
    F.row_number().over(janela_genero).cast("bigint").alias("sk_genre_id"),
    F.col("nome_genero").cast("string"),
)

dim_genres.write.mode("overwrite").saveAsTable(f"{schema_gold}.dim_genres")

display(spark.table(f"{schema_gold}.dim_genres"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_genre_id,nome_genero
1,Action
2,Adventure
3,Animation
4,Comedy
5,Crime
6,Documentary
7,Drama
8,Family
9,Fantasy
10,History


In [0]:
pessoas_empresas = spark.table(f"{schema_silver}.tb_pessoas_empresas")

# dim_people: só pessoas físicas, um registro por par (nome, tipo)
catalogo_pessoas = (
    pessoas_empresas
        .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
        .select(
            F.col("nome_entidade").alias("nome_pessoa"),
            F.col("tipo_entidade").alias("tipo_pessoa"),
        )
        .filter(F.col("nome_pessoa").isNotNull())
        .distinct()
)

janela_pessoa = Window.orderBy("nome_pessoa", "tipo_pessoa")

dim_people = catalogo_pessoas.select(
    F.row_number().over(janela_pessoa).cast("bigint").alias("sk_person_id"),
    F.col("nome_pessoa").cast("string"),
    F.col("tipo_pessoa").cast("string"),
)

dim_people.write.mode("overwrite").saveAsTable(f"{schema_gold}.dim_people")


# dim_companies: só produtoras, um registro por nome
catalogo_produtoras = (
    pessoas_empresas
        .filter(F.col("tipo_entidade") == "Produtora")
        .select(F.col("nome_entidade").alias("nome_produtora"))
        .filter(F.col("nome_produtora").isNotNull())
        .distinct()
)

janela_produtora = Window.orderBy("nome_produtora")

dim_companies = catalogo_produtoras.select(
    F.row_number().over(janela_produtora).cast("bigint").alias("sk_company_id"),
    F.col("nome_produtora").cast("string"),
)

dim_companies.write.mode("overwrite").saveAsTable(f"{schema_gold}.dim_companies")


print("pessoas:", spark.table(f"{schema_gold}.dim_people").count())
print("produtoras:", spark.table(f"{schema_gold}.dim_companies").count())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


pessoas: 417527
produtoras: 44431


In [0]:
dim_movies_chaves = spark.table(f"{schema_gold}.dim_movies").select("sk_movie_id", "id_filme")
dim_genres_chaves = spark.table(f"{schema_gold}.dim_genres")

bridge_movie_genre = (
    spark.table(f"{schema_silver}.tb_generos")
        .join(dim_movies_chaves, on="id_filme", how="inner")
        .join(dim_genres_chaves, on="nome_genero", how="inner")
        .select(
            F.col("sk_movie_id").cast("bigint"),
            F.col("sk_genre_id").cast("bigint"),
        )
        .distinct()
)

bridge_movie_genre.write.mode("overwrite").saveAsTable(f"{schema_gold}.bridge_movie_genre")

print("pares filme-gênero:", spark.table(f"{schema_gold}.bridge_movie_genre").count())

pares filme-gênero: 140078


In [0]:
pessoas_empresas = spark.table(f"{schema_silver}.tb_pessoas_empresas")

# bridge_movie_person: liga filme a pessoa pelo par (nome, tipo)
bridge_movie_person = (
    pessoas_empresas
        .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
        .select(
            "id_filme",
            F.col("nome_entidade").alias("nome_pessoa"),
            F.col("tipo_entidade").alias("tipo_pessoa"),
        )
        .join(dim_movies_chaves, on="id_filme", how="inner")
        .join(spark.table(f"{schema_gold}.dim_people"), on=["nome_pessoa", "tipo_pessoa"], how="inner")
        .select(
            F.col("sk_movie_id").cast("bigint"),
            F.col("sk_person_id").cast("bigint"),
        )
        .distinct()
)

bridge_movie_person.write.mode("overwrite").saveAsTable(f"{schema_gold}.bridge_movie_person")


# bridge_movie_company: liga filme a produtora pelo nome
bridge_movie_company = (
    pessoas_empresas
        .filter(F.col("tipo_entidade") == "Produtora")
        .select(
            "id_filme",
            F.col("nome_entidade").alias("nome_produtora"),
        )
        .join(dim_movies_chaves, on="id_filme", how="inner")
        .join(spark.table(f"{schema_gold}.dim_companies"), on="nome_produtora", how="inner")
        .select(
            F.col("sk_movie_id").cast("bigint"),
            F.col("sk_company_id").cast("bigint"),
        )
        .distinct()
)

bridge_movie_company.write.mode("overwrite").saveAsTable(f"{schema_gold}.bridge_movie_company")


print("pares filme-pessoa:", spark.table(f"{schema_gold}.bridge_movie_person").count())
print("pares filme-produtora:", spark.table(f"{schema_gold}.bridge_movie_company").count())

pares filme-pessoa: 754365
pares filme-produtora: 114635


In [0]:
avaliacoes = spark.table(f"{schema_silver}.tb_avaliacoes_usuarios")

# Uma linha por filme: total de avaliações e média das notas válidas
resumo = (
    avaliacoes
        .groupBy("id_filme")
        .agg(
            F.count("*").alias("qtd_avaliacoes_usuarios"),
            F.round(F.avg("nota_usuario"), 2).alias("nota_media_usuarios"),
        )
)

# Traduz id_filme para sk_movie_id
resumo = resumo.join(dim_movies_chaves, on="id_filme", how="inner")

janela_review = Window.orderBy("sk_movie_id")

dim_reviews = resumo.select(
    F.row_number().over(janela_review).cast("bigint").alias("sk_review_id"),
    F.col("sk_movie_id").cast("bigint"),
    F.col("qtd_avaliacoes_usuarios").cast("int"),
    F.col("nota_media_usuarios").cast("double"),
)

dim_reviews.write.mode("overwrite").saveAsTable(f"{schema_gold}.dim_reviews")

display(spark.table(f"{schema_gold}.dim_reviews"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
1,1,1,0.1
2,5,1,3.3
3,8,1,6.8
4,17,1,5.4
5,19,1,5.3
6,20,1,4.4
7,23,3,4.75
8,24,1,4.8
9,26,1,6.6
10,33,1,7.5


In [0]:
# Receita total de todos os filmes somados
display(
    spark.table(f"{schema_gold}.fact_movies_performance")
        .agg(F.sum("receita_brl").alias("receita_total_brl"))
)

receita_total_brl
806537380275.18


In [0]:
# Os 5 filmes com maior popularidade

display(
    spark.table(f"{schema_gold}.fact_movies_performance")
    .join(spark.table(f"{schema_gold}.dim_movies"), on="sk_movie_id", how="inner")
        .select("titulo", "popularidade")
        .orderBy(F.desc("popularidade"))
        .limit(5)
)

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
The Nun II,1692.778
Meg 2: The Trench,1567.273
retribution,1547.22


In [0]:
# quantidade de filmes por gênero, do maior para o menor
display(
    spark.table(f"{schema_gold}.bridge_movie_genre")
        .join(spark.table(f"{schema_gold}.dim_genres"), on="sk_genre_id", how="inner")
        .groupBy("nome_genero")
        .agg(F.count("*").alias("qtd_filmes"))
        .orderBy(F.desc("qtd_filmes"))
)

nome_genero,qtd_filmes
Drama,32196
Documentary,18943
Comedy,18572
Thriller,10255
Horror,9704
Romance,7625
Action,6039
Crime,4731
Animation,4459
TV Movie,4073


In [0]:
# Pergunta 4: top 10 filmes por receita, com posição no ranking (RANK)
janela_receita = Window.orderBy(F.desc("receita_usd"))

display(
    spark.table(f"{schema_gold}.fact_movies_performance")
        .filter(F.col("receita_usd").isNotNull())
        .join(spark.table(f"{schema_gold}.dim_movies"), on="sk_movie_id", how="inner")
        .withColumn("posicao", F.rank().over(janela_receita))
        .filter(F.col("posicao") <= 10)
        .select("posicao", "titulo", "receita_usd", "receita_brl")
        .orderBy("posicao")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


posicao,titulo,receita_usd,receita_brl
1,Avengers: Endgame,2800000000.00,13913760000.00
2,Avatar: The Way of Water,2320250281.00,11529787696.35
3,AVENGERS: INFINITY WAR,2052415039.00,10198860811.80
4,spider-man: no way home,1921847111.00,9550042663.98
5,The Lion King,1663075401.00,8264154282.65
6,Top Gun: Maverick,1488732821.00,7397811134.11
7,Barbie,1428545028.00,7098725953.14
8,The Super Mario Bros. Movie,1355725263.00,6736869976.90
9,Black Panther,1349926083.00,6708052691.64
10,Star Wars: The Last Jedi,1332698830.00,6622447026.04


In [0]:
# serve para achar o filme mais recente lançado no bd, pra usar nas questões 5 e 6
data_limite = (
    spark.table(f"{schema_gold}.dim_movies")
        .filter(F.col("status_filme") == "Lançado")
        .filter(F.col("data_lancamento") <= F.current_date())
        .agg(F.max("data_lancamento"))
        .first()[0]
)

print("data limite:", data_limite)

data limite: 2026-02-19


In [0]:
# ator ou atriz com mais participações nos últimos 2 anos, levando em consideração a data limite da célula anterior
inicio_2_anos = F.add_months(F.lit(data_limite), -24)

participacoes = (
    spark.table(f"{schema_gold}.bridge_movie_person")
        .join(spark.table(f"{schema_gold}.dim_people"), on="sk_person_id", how="inner")
        .filter(F.col("tipo_pessoa") == "Ator")
        .join(spark.table(f"{schema_gold}.dim_movies"), on="sk_movie_id", how="inner")
        .filter(F.col("status_filme") == "Lançado")
        .filter(F.col("data_lancamento") > inicio_2_anos)
        .filter(F.col("data_lancamento") <= F.lit(data_limite))
        .groupBy("nome_pessoa")
        .agg(F.countDistinct(F.lower(F.trim(F.col("titulo")))).alias("qtd_participacoes"))
)

janela_ator = Window.orderBy(F.desc("qtd_participacoes"))

display(
    participacoes
        .withColumn("posicao", F.rank().over(janela_ator))
        .filter(F.col("posicao") == 1)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


nome_pessoa,qtd_participacoes,posicao
Kevin Hart,7,1


In [0]:
# Pergunta 6: produtora com maior lucro somado nos últimos 5 anos (até a data limite)
inicio_5_anos = F.add_months(F.lit(data_limite), -60)

filmes_por_produtora = (
    spark.table(f"{schema_gold}.bridge_movie_company")
        .join(spark.table(f"{schema_gold}.dim_companies"), on="sk_company_id", how="inner")
        .join(spark.table(f"{schema_gold}.dim_movies"), on="sk_movie_id", how="inner")
        .filter(F.col("status_filme") == "Lançado")
        .filter(F.col("data_lancamento") > inicio_5_anos)
        .filter(F.col("data_lancamento") <= F.lit(data_limite))
        .join(spark.table(f"{schema_gold}.fact_movies_performance"), on="sk_movie_id", how="inner")
        .withColumn("titulo_normalizado", F.lower(F.trim(F.col("titulo"))))
        # 1ª agregação: uma linha por (produtora, filme), colapsando as cópias
        .groupBy("nome_produtora", "titulo_normalizado")
        .agg(
            F.max("lucro_usd").alias("lucro_usd"),
            F.max("lucro_brl").alias("lucro_brl"),
        )
)

# 2ª agregação: lucro total por produtora
lucro_produtoras = (
    filmes_por_produtora
        .groupBy("nome_produtora")
        .agg(
            F.sum("lucro_usd").alias("lucro_total_usd"),
            F.sum("lucro_brl").alias("lucro_total_brl"),
        )
)

janela_produtora_lucro = Window.orderBy(F.desc("lucro_total_usd"))

display(
    lucro_produtoras
        .withColumn("posicao", F.rank().over(janela_produtora_lucro))
        .filter(F.col("posicao") == 1)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


nome_produtora,lucro_total_usd,lucro_total_brl,posicao
Universal Pictures,5772329679.00,28683860640.89,1
